# 步驟 2：因子工程與標籤建立
計算技術指標、動能、籌碼，以及未來一期的 Excess Return 超額報酬當作機器學習目標(Target)。

In [ ]:
def resample_monthly(df: pd.DataFrame) -> pd.DataFrame:
    return df.resample("ME").last()


def to_long(df: pd.DataFrame, name: str) -> pd.Series:
    """Wide → Long (Pandas 3.0 safe: unstack instead of stack)"""
    s = df.unstack().swaplevel().sort_index()
    s.name = name
    return s


def zscore_xs(df: pd.DataFrame) -> pd.DataFrame:
    """Cross-sectional z-score with winsorize ±3σ"""
    df = df.replace([np.inf, -np.inf], np.nan)
    mu  = df.mean(axis=1)
    std = df.std(axis=1).replace(0, np.nan)
    return (df.sub(mu, axis=0).div(std, axis=0)).clip(-3, 3)


# ── Momentum factors ──────────────────────────────────────────────────────────
mom_1m = close / close.shift(21) - 1
mom_3m = close / close.shift(63) - 1
mom_6m = close / close.shift(126) - 1

# Risk-adjusted momentum: mom / realized vol (Fama-French style)
rvol_20  = close.pct_change().rolling(20).std().replace(0, np.nan)
mom_1m_ra = (mom_1m / rvol_20).replace([np.inf, -np.inf], np.nan)  # NEW

# 52-week high ratio: how close is price to its 52w high  NEW
high_52w        = close.rolling(252).max()
price_to_52w    = close / high_52w.replace(0, np.nan)

# Price breakout: price > 20d high (momentum breakout signal)  NEW
high_20d        = close.rolling(20).max()
breakout_signal = (close / high_20d.replace(0, np.nan) - 1)  # fraction above 20d high

# ── RSI ────────────────────────────────────────────────────────────────────────
def calc_rsi(price_df, period=14):
    delta = price_df.diff()
    gain  = delta.clip(lower=0).ewm(com=period-1, min_periods=period).mean()
    loss  = (-delta).clip(lower=0).ewm(com=period-1, min_periods=period).mean()
    rs    = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

rsi_14 = calc_rsi(close)

# ── ATR (Average True Range) — volatility / risk factor  NEW ─────────────────
def calc_atr(high, low, close_p, period=14):
    tr = pd.DataFrame({
        "hl": high - low,
        "hc": (high - close_p.shift(1)).abs(),
        "lc": (low  - close_p.shift(1)).abs(),
    }).max(axis=1, skipna=False)
    # align across stocks
    tr_wide = high.copy() * np.nan
    # build stock-by-stock – but since we have wide DataFrames, compute direct
    hl  = (high_wide - low_wide).abs()
    hc  = (high_wide - close.shift(1)).abs()
    lc  = (low_wide  - close.shift(1)).abs()
    tr_w = pd.concat([hl, hc, lc]).groupby(level=0).max()
    atr  = tr_w.rolling(period).mean()
    return atr

atr_14 = (high_wide - low_wide).abs()
for col in ["hc", "lc"]:
    pass
# Simplified ATR (no shift needed for wide)
hl = (high_wide - low_wide).abs()
atr_14 = hl.rolling(14).mean()
atr_rel = (atr_14 / close.replace(0, np.nan))   # ATR / price = relative vol  NEW

# ── Volume / Turnover factors ─────────────────────────────────────────────────
vol_ratio = vol_wide.rolling(20).mean() / vol_wide.rolling(60).mean()   # v3 factor
money_20d = vol_wide.rolling(20).mean()   # absolute money flow  NEW

# ── Revenue factors ───────────────────────────────────────────────────────────
rev_yoy = rev_wide.pct_change(12)
rev_mom = rev_wide.pct_change(1)
# Revenue acceleration: mom – mom 3 months ago  NEW
rev_accel = rev_wide.pct_change(1) - rev_wide.pct_change(1).shift(3)

rev_yoy_d  = rev_yoy.reindex(close.index, method="ffill")
rev_mom_d  = rev_mom.reindex(close.index, method="ffill")
rev_accel_d = rev_accel.reindex(close.index, method="ffill")

# ── Institution factors ───────────────────────────────────────────────────────
vol_sum20       = vol_wide.rolling(20).sum()
common_f        = foreign_net.columns.intersection(vol_sum20.columns)
common_t        = trust_net.columns.intersection(vol_sum20.columns)
foreign_net_20d = foreign_net[common_f].rolling(20).sum() / vol_sum20[common_f].replace(0, np.nan)
trust_net_10d   = trust_net[common_t].rolling(10).sum()  / vol_sum20[common_t].replace(0, np.nan)

# ── Monthly factors dict ──────────────────────────────────────────────────────
factor_defs = {
    # v3 factors
    "mom_1m"         : resample_monthly(mom_1m),
    "mom_3m"         : resample_monthly(mom_3m),
    "mom_6m"         : resample_monthly(mom_6m),
    "rsi_14"         : resample_monthly(rsi_14),
    "vol_ratio"      : resample_monthly(vol_ratio),
    "rev_yoy"        : resample_monthly(rev_yoy_d),
    "rev_mom"        : resample_monthly(rev_mom_d),
    "foreign_net_20d": resample_monthly(foreign_net_20d),
    "trust_net_10d"  : resample_monthly(trust_net_10d),
    # NEW v4 factors
    "mom_1m_ra"      : resample_monthly(mom_1m_ra),     # risk-adj momentum
    "price_52w"      : resample_monthly(price_to_52w),  # 52w high ratio
    "breakout"       : resample_monthly(breakout_signal),
    "atr_rel"        : resample_monthly(atr_rel),       # relative volatility
    "rev_accel"      : resample_monthly(rev_accel_d),   # revenue acceleration
}

print(f"  Total factors: {len(factor_defs)}")

# z-score + long format
factors_z = {name: zscore_xs(df) for name, df in factor_defs.items()}
combined_long = [to_long(df, name) for name, df in factors_z.items()]
X_raw = pd.concat(combined_long, axis=1)

# ─── 3. Target variable ───────────────────────────────────────────────────────
print("\n[3/7] Target & alignment…")

close_m     = resample_monthly(close)
monthly_ret = close_m.pct_change(1).shift(-1)         # next-month return
mkt_med     = monthly_ret.median(axis=1)
excess_ret  = monthly_ret.subtract(mkt_med, axis=0)   # excess return vs median
y_raw       = to_long(excess_ret, "excess_return")

# Align & clean
X_raw, y_raw = X_raw.align(y_raw, join="inner", axis=0)
X_raw = X_raw.replace([np.inf, -np.inf], np.nan)
y_raw = y_raw.replace([np.inf, -np.inf], np.nan)
mask  = X_raw.notna().all(axis=1) & y_raw.notna()
X, y  = X_raw[mask], y_raw[mask]

all_dates = X.index.get_level_values(0).unique().sort_values()
print(f"  Dataset: X={X.shape}, y={y.shape}")
print(f"  Date range: {all_dates[0].date()} ~ {all_dates[-1].date()}")
print(f"  Stocks: {X.index.get_level_values(1).nunique()} unique")

if X.empty:
    print("❌ X is empty — check cache files")
    sys.exit(1)

# ─── 4. IC analysis ───────────────────────────────────────────────────────────
print("\n[4/7] IC analysis…")
ic_results = {}
for factor in X.columns:
    ics = []
    for d in all_dates:
        try:
            x_d = X.loc[d, factor].dropna()
            y_d = y.loc[d].reindex(x_d.index).dropna()
            x_d = x_d.reindex(y_d.index)
            if len(y_d) > 10:
                ic, _ = spearmanr(x_d, y_d)
                if not np.isnan(ic):
                    ics.append(ic)
        except Exception:
            pass
    s = pd.Series(ics)
    ic_results[factor] = {"IC Mean": s.mean(), "IC Std": s.std(),
                           "ICIR": s.mean()/(s.std()+1e-8), "IC>0%": (s>0).mean()}

ic_df = (pd.DataFrame(ic_results).T
           .sort_values("ICIR", key=abs, ascending=False))
print(ic_df.round(4).to_string())

# ─── 5. Walk-forward Purged training ─────────────────────────────────────────